# Lesson 9b: From Raw Text & Images to Predictions (worked examples)

This notebook shows the FULL journey, with every step printed:

**raw input  ->  turn into numbers (features)  ->  feed to model  ->  get prediction**

Part A: TEXT (movie reviews -> positive/negative)
Part B: IMAGES (handwritten digits -> which number)

> Upload to [Google Colab](https://colab.research.google.com) and run top to bottom.

# PART A: TEXT

## A1: The raw input — what the 'question' looks like

Just plain English sentences and a label (1 = positive, 0 = negative).

In [ ]:
reviews = [
    'this movie was great and fun',      # positive
    'fantastic film loved it',           # positive
    'great acting wonderful story',      # positive
    'terrible movie boring and bad',     # negative
    'awful film hated it',               # negative
    'bad acting boring story',           # negative
]
labels = [1, 1, 1, 0, 0, 0]

for r, l in zip(reviews, labels):
    print(f'[{"POS" if l==1 else "NEG"}]  "{r}"')

## A2: Transform text -> numbers (Bag of Words)

`CountVectorizer` builds a vocabulary of all words, then counts each word per review. Watch the text become a numeric table.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(reviews)   # the transformation!

# Show it as a readable table: one column per word, counts per review
bag = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print('Vocabulary (all words found):')
print(list(vectorizer.get_feature_names_out()))
print('\nThe numeric table the model actually sees:')
print(bag)

Each row is now a list of numbers (word counts) — exactly the kind of table every model can use. The text is gone; only numbers remain.

## A3: Feed to a model and train

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X, labels)   # same .fit() pattern as always
print('Model trained on the word-count table.')

## A4: Predict on a NEW sentence

Key point: the new text must be transformed the SAME way (same vocabulary) before predicting.

In [ ]:
new_reviews = ['great fun film', 'boring and awful']

new_X = vectorizer.transform(new_reviews)   # SAME vectorizer, just transform (not fit)
preds = model.predict(new_X)
probs = model.predict_proba(new_X)[:, 1]

for text, pred, prob in zip(new_reviews, preds, probs):
    print(f'"{text}"  ->  {"POSITIVE" if pred==1 else "NEGATIVE"}  (P(positive)={prob:.2f})')

## A5: TF-IDF version (down-weights common words)

Same idea, but instead of raw counts it uses TF-IDF scores. Notice the numbers are now decimals (weights), not whole counts.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(reviews)

tfidf_table = pd.DataFrame(X_tfidf.toarray().round(2), columns=tfidf.get_feature_names_out())
print('TF-IDF table (decimals = importance weights, not counts):')
print(tfidf_table)

# PART B: IMAGES

## B1: The raw input — an image IS a grid of numbers

We use the built-in 'digits' dataset: 8x8 grayscale images of handwritten numbers 0-9.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()

# Look at the first image BOTH as a picture and as its raw numbers
first_image = digits.images[0]
print(f'This image is labeled: {digits.target[0]}')
print(f'Shape: {first_image.shape}  (an 8x8 grid)')
print('\nThe raw pixel numbers (0=dark, 16=bright):')
print(first_image.astype(int))

plt.imshow(first_image, cmap='gray')
plt.title(f'Label: {digits.target[0]}')
plt.show()

## B2: Transform — flatten the 8x8 grid into 64 features

We lay the grid out into one long row of 64 numbers. Now each image is just a row of features, like any table.

In [ ]:
X = digits.data      # already flattened: each row is 64 numbers
y = digits.target    # the correct digit 0-9

print(f'Dataset shape: {X.shape}  (1797 images, each 64 features)')
print(f'\nFirst image as 64 features:')
print(X[0].astype(int))

## B3: Feed to model, train, and predict

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
print(f'Test accuracy: {acc:.3f}  (predicting the right digit)')

In [ ]:
# Show a few test images with the model's prediction
fig, axes = plt.subplots(1, 5, figsize=(10, 3))
for ax, img, true in zip(axes, X_test[:5], y_test[:5]):
    pred = model.predict([img])[0]
    ax.imshow(img.reshape(8, 8), cmap='gray')
    ax.set_title(f'pred: {pred}\ntrue: {true}')
    ax.axis('off')
plt.show()

## The takeaway

- **Text**: `CountVectorizer` / `TfidfVectorizer` turn sentences into a numeric table (word counts / weights), then the SAME `.fit()` / `.predict()` pattern works.
- **Images**: each image is a grid of pixel numbers; flatten it into a row of features, then the SAME pattern works.
- **The model never sees text or pixels — only numbers.** Feature extraction is the bridge.
- For complex images/text, deep learning (Part D) learns the feature extraction automatically.

## Your turn

1. Add your own new sentence in A4 and see the prediction.
2. In A2, what number appears in the 'great' column for the review 'great acting wonderful story'?
3. In B1, change the index from 0 to 5. What digit is it, and does the pixel grid look like that digit?
4. Why must we use the SAME vectorizer (transform, not fit) on new text?